# Assignment 5 Option C: LM Evaluation Report

## Task Proposal: Social Media Sentiment and Topic Classification

### Task Description

This project evaluates how well different language models can classify short social media posts by both sentiment and business topic. Each input is a single tweet from the Brands and Product Emotions dataset, which contains real social media posts about brands and products.

The expected output has two parts:

1. **Sentiment label**: `positive`, `negative`, or `neutral`
2. **Topic label**: the brand or product that the post is mainly about

The model should return its answer in structured JSON format:

```json
{
  "sentiment": "negative",
  "topic": "iPhone"
}
```

Each model output will be compared against ground truth labels from the selected test set. The original dataset labels are used as a starting point, and the final 30-example test set should be manually reviewed before running the full evaluation.

### Why This Matters for Business

Social media posts give businesses fast feedback about customer satisfaction, product problems, service issues, and brand perception. However, manually reading and labeling large numbers of posts is time-consuming. If language models can accurately identify both sentiment and topic, businesses can monitor customer opinion, detect recurring problems, prioritize support responses, and understand which brands or products are creating positive or negative reactions.

For example, a company could quickly see whether negative posts are mostly about an app, a device, a brand experience, or a product launch. This can help managers make better decisions about marketing, customer support, product improvement, and brand reputation.

### Success Criteria

A good model output should meet the following criteria:

1. Correctly classify the overall sentiment as `positive`, `negative`, or `neutral`.
2. Correctly identify the main brand or product topic from the approved topic list.
3. Return valid JSON with both required fields: `sentiment` and `topic`.
4. Handle ambiguous or mixed posts by choosing the dominant sentiment and main topic.
5. Avoid adding unsupported information that is not present in the original post.

Model performance will be evaluated using sentiment accuracy, topic accuracy, exact-match accuracy, format compliance, and average latency. Exact-match accuracy means both the sentiment and topic labels are correct for the same post.

## 1. Setup

This notebook uses the Kaggle `Brands and Product Emotions` dataset. If the raw CSV is missing, the next cell tries to download it with `kagglehub`.

In [1]:
from pathlib import Path
import json
import os
import re
import time
import html

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
EVALUATION_DIR = PROJECT_ROOT / "evaluation"

for folder in [DATA_DIR, RAW_DIR, RESULTS_DIR, PROMPTS_DIR, EVALUATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "judge-1377884607_tweet_product_company.csv"
TEST_SET_FILE = DATA_DIR / "test_set.csv"
RESULTS_FILE = RESULTS_DIR / "raw_model_outputs.csv"
MATRIX_FILE = RESULTS_DIR / "results_matrix.csv"

print(PROJECT_ROOT)

/Users/anthonyhanna/Documents/New project/bsan6200-assignment5


In [2]:
if not RAW_FILE.exists():
    try:
        import kagglehub
        downloaded_path = Path(kagglehub.dataset_download("drishtiagarwal20/brands-and-product-emotions"))
        source_csv = next(downloaded_path.glob("*.csv"))
        RAW_FILE.write_bytes(source_csv.read_bytes())
        print(f"Downloaded dataset to {RAW_FILE}")
    except Exception as exc:
        raise RuntimeError(
            "Could not download the Kaggle dataset automatically. "
            "Download it manually from https://www.kaggle.com/datasets/drishtiagarwal20/brands-and-product-emotions "
            f"and place the CSV at {RAW_FILE}"
        ) from exc
else:
    print(f"Raw dataset already exists: {RAW_FILE}")

Raw dataset already exists: /Users/anthonyhanna/Documents/New project/bsan6200-assignment5/data/raw/judge-1377884607_tweet_product_company.csv


## 2. Load and Clean the Dataset

The original dataset has three useful columns: tweet text, brand/product target, and emotion toward the brand/product.

In [3]:
raw_df = pd.read_csv(RAW_FILE, encoding="latin1")
raw_df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [4]:
raw_df.shape, raw_df.columns.tolist()

((9093, 3),
 ['tweet_text',
  'emotion_in_tweet_is_directed_at',
  'is_there_an_emotion_directed_at_a_brand_or_product'])

In [5]:
def clean_text(text):
    text = html.unescape(str(text))
    text = text.encode("ascii", errors="ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text

sentiment_map = {
    "Positive emotion": "positive",
    "Negative emotion": "negative",
    "No emotion toward brand or product": "neutral",
}

df = raw_df.rename(columns={
    "tweet_text": "post",
    "emotion_in_tweet_is_directed_at": "topic",
    "is_there_an_emotion_directed_at_a_brand_or_product": "raw_sentiment",
}).copy()

df = df[df["topic"].notna()].copy()
df = df[df["raw_sentiment"].isin(sentiment_map)].copy()
df["sentiment"] = df["raw_sentiment"].map(sentiment_map)
df["post"] = df["post"].map(clean_text)
df = df[df["post"].str.len().between(30, 260)].copy()

df[["post", "sentiment", "topic"]].head()

,post,sentiment,topic
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,negative,iPhone
1,@jessedee Know about @fludapp ? Awesome iPad/i...,positive,iPad or iPhone App
2,@swonderlin Can not wait for #iPad 2 also. The...,positive,iPad
3,@sxsw I hope this year's festival isn't as cra...,negative,iPad or iPhone App
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,positive,Google


In [6]:
print("Clean rows available:", len(df))
print("\nSentiment counts:")
print(df["sentiment"].value_counts())
print("\nTopic counts:")
print(df["topic"].value_counts())

Clean rows available: 3278

Sentiment counts:
sentiment
positive    2668
negative     519
neutral       91
Name: count, dtype: int64

Topic counts:
topic
iPad                               939
Apple                              659
iPad or iPhone App                 470
Google                             429
iPhone                             296
Other Google product or service    291
Android App                         81
Android                             78
Other Apple product or service      35
Name: count, dtype: int64


## 3. Create the 30-Example Test Set

The assignment requires 30+ examples with ground truth. This cell creates a balanced 30-row test set: 10 positive, 10 negative, and 10 neutral posts.

Before final submission, manually review `data/test_set.csv` and adjust any labels you disagree with. That review step makes the final ground truth yours.

In [7]:
samples = []
for sentiment, n, seed in [("positive", 10, 42), ("negative", 10, 43), ("neutral", 10, 44)]:
    subset = df[df["sentiment"] == sentiment].sample(frac=1, random_state=seed).head(n)
    samples.append(subset)

test_set = pd.concat(samples).reset_index(drop=True)
test_set.insert(0, "id", range(1, len(test_set) + 1))
test_set["case_type"] = ["standard"] * 20 + ["ambiguous_hard"] * 5 + ["edge"] * 5
test_set = test_set[["id", "post", "sentiment", "topic", "case_type"]]
test_set.to_csv(TEST_SET_FILE, index=False)
test_set

,id,post,sentiment,topic,case_type
0,1,This dude next to me is holding and using an i...,positive,iPad,standard
1,2,RT @mention I hope everyone has an awesome wee...,positive,Apple,standard
2,3,RT @mention line moving fast! Rt @mention have...,positive,Apple,standard
3,4,Last minute flight change to #sxsw means I can...,positive,iPad,standard
4,5,YES RT @mention LoL U r gadgetzilla! Have fun!...,positive,iPad,standard
5,6,"bored at #SxSW, try this iphone app: {link}",positive,iPad or iPhone App,standard
6,7,Holler Gram for iPad on the iTunes App Store: ...,positive,iPad or iPhone App,standard
7,8,All this #sxsw gadget lust is rubbing off on m...,positive,iPad,standard
8,9,"#GoogleDoodle #sxsw Google Doodles, simple on ...",positive,Other Google product or service,standard
9,10,keep up with new on #Japan quake from iPhone a...,positive,iPhone,standard


In [8]:
TOPICS = sorted(test_set["topic"].unique())
SENTIMENTS = ["positive", "negative", "neutral"]

print("Allowed sentiments:", SENTIMENTS)
print("Allowed topics:")
for topic in TOPICS:
    print("-", topic)

Allowed sentiments: ['positive', 'negative', 'neutral']
Allowed topics:
- Android
- Apple
- Google
- Other Google product or service
- iPad
- iPad or iPhone App
- iPhone


## 4. Prompt Templates

This project tests three prompting strategies for each model: zero-shot, few-shot, and structured JSON.

In [9]:
def topic_list_text():
    return ", ".join(TOPICS)

PROMPT_TEMPLATES = {
    "zero_shot": """Classify the following social media post.

Allowed sentiment labels: positive, negative, neutral.
Allowed topic labels: {topics}.

Post: {post}

Return only JSON with this schema:
{{"sentiment": "...", "topic": "..."}}""",

    "few_shot": """Classify the social media post by sentiment and topic.

Allowed sentiment labels: positive, negative, neutral.
Allowed topic labels: {topics}.

Examples:
Post: RT @mention I hope everyone has an awesome weekend at #SXSW! I know @mention is giving away some great Apple prizes.
Output: {{"sentiment": "positive", "topic": "Apple"}}

Post: @mention really disappointed with the iPad app - lots of error messages have to switch to tweet deck for the rest of #sxsw
Output: {{"sentiment": "negative", "topic": "iPad or iPhone App"}}

Post: Check out iPad Design Headaches (2 Tablets, Call in the Morning) at SXSW. {{link}} #SXSW #tapworthy
Output: {{"sentiment": "neutral", "topic": "iPad"}}

Now classify this post:
Post: {post}
Output:""",

    "structured_json": """You are a business analyst classifying brand-related social media posts.

Choose exactly one sentiment from: positive, negative, neutral.
Choose exactly one topic from: {topics}.

Rules:
- Use positive when the post expresses approval, excitement, appreciation, or praise.
- Use negative when the post expresses frustration, criticism, disappointment, or dislike.
- Use neutral when the post mostly reports information, asks a question, or mentions a brand/product without clear emotion.
- Do not invent a topic. Pick the closest topic from the allowed list.
- Return valid JSON only. No markdown. No explanation.

Post: {post}

JSON schema:
{{"sentiment": "positive|negative|neutral", "topic": "one allowed topic"}}""",
}


def build_prompt(strategy, post):
    return PROMPT_TEMPLATES[strategy].format(post=post, topics=topic_list_text())

def extract_json(text):
    if not isinstance(text, str):
        return {}
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    # Fallback for small local models that return terse text instead of JSON.
    lower = text.lower()
    sentiment = ""
    for candidate in SENTIMENTS:
        if candidate in lower:
            sentiment = candidate
            break
    topic = ""
    for candidate in TOPICS:
        if candidate.lower() in lower:
            topic = candidate
            break
    if sentiment or topic:
        return {"sentiment": sentiment, "topic": topic}
    return {}

def normalize_label(value):
    return str(value).strip() if pd.notna(value) else ""

prompt_doc = ["# Prompt Templates", ""]
for name, template in PROMPT_TEMPLATES.items():
    prompt_doc.extend([f"## {name}", "", "```text", template, "```", ""])
(PROMPTS_DIR / "prompt_templates.md").write_text("\n".join(prompt_doc), encoding="utf-8")
print((PROMPTS_DIR / "prompt_templates.md"))

/Users/anthonyhanna/Documents/New project/bsan6200-assignment5/prompts/prompt_templates.md


## 5. Model Setup

The assignment requires at least two different models. This notebook compares one Google model and one Hugging Face model:

1. **Google Gemini API**: `gemini-flash-lite-latest`
2. **Hugging Face local Transformers model**: `typeform/distilbert-base-uncased-mnli`

The Hugging Face Inference API route was tested, but the current token returned a monthly-credit-depleted error. To keep the project free while still using a Hugging Face model, this notebook runs a local Hugging Face zero-shot classifier with `transformers`. This is much faster than local text generation and works well for the assignment's sentiment + topic label task.

Create a `.env` file in the project root with whichever keys you use:

```text
GOOGLE_API_KEY=your_google_ai_studio_key
HF_TOKEN=your_huggingface_read_token
```

Set `RUN_API_CALLS = True` only when you are ready to run the full experiment.

In [10]:
RUN_API_CALLS = True

# Required: use at least two models. This project uses one Google model and one Hugging Face model.
MODELS = [
    {
        "provider": "gemini",
        "model": os.getenv("GEMINI_MODEL", "gemini-flash-lite-latest"),
        "access_method": "Google AI Studio API, free tier",
        "cost_per_call": 0,
    },
    {
        "provider": "hf_local",
        "model": os.getenv("HF_LOCAL_MODEL", "typeform/distilbert-base-uncased-mnli"),
        "access_method": "Hugging Face Transformers local zero-shot classifier",
        "cost_per_call": 0,
    },
]

# Optional backup choices from the assignment guideline if available:
# MODELS.append({
#     "provider": "huggingface",
#     "model": "meta-llama/Llama-3.1-8B-Instruct",
#     "access_method": "Hugging Face Inference API / Inference Providers, free token",
#     "cost_per_call": 0,
# })
# MODELS.append({
#     "provider": "ollama",
#     "model": "llama3",
#     "access_method": "Ollama local",
#     "cost_per_call": 0,
# })

pd.DataFrame(MODELS)

,provider,model,access_method,cost_per_call
0,gemini,gemini-flash-lite-latest,"Google AI Studio API, free tier",0
1,hf_local,typeform/distilbert-base-uncased-mnli,Hugging Face Transformers local zero-shot clas...,0


In [11]:
def call_gemini(model_name, prompt):
    import google.generativeai as genai
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing GOOGLE_API_KEY. Add it to .env before running Gemini calls.")

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(model_name)

    last_error = None
    for attempt in range(3):
        start = time.perf_counter()
        try:
            response = model.generate_content(prompt)
            latency = time.perf_counter() - start
            # Free-tier Gemini allows about 10 requests/minute for this model.
            time.sleep(6.5)
            return response.text, latency
        except Exception as exc:
            last_error = exc
            if "429" in str(exc) and attempt < 2:
                time.sleep(65)
                continue
            raise
    raise last_error

def call_huggingface(model_name, prompt):
    from dotenv import load_dotenv
    from huggingface_hub import InferenceClient

    load_dotenv(PROJECT_ROOT / ".env")
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if not token:
        raise RuntimeError("Missing HF_TOKEN. Add a Hugging Face READ token to .env before running HF calls.")

    client = InferenceClient(api_key=token)
    start = time.perf_counter()
    response = client.chat_completion(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120,
        temperature=0.1,
    )
    latency = time.perf_counter() - start
    return response.choices[0].message.content, latency


_HF_LOCAL_PIPELINES = {}

_HF_LOCAL_MODELS = {}

def call_hf_local(model_name, prompt):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    if model_name not in _HF_LOCAL_MODELS:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        _HF_LOCAL_MODELS[model_name] = (tokenizer, model)

    tokenizer, model = _HF_LOCAL_MODELS[model_name]
    compact_prompt = (
        "Classify this social media post. "
        "Sentiment choices: positive, negative, neutral. "
        f"Topic choices: {topic_list_text()}. "
        "Answer exactly as JSON with sentiment and topic. "
        + prompt.split("Post:")[-1].strip()[:600]
    )
    start = time.perf_counter()
    inputs = tokenizer(compact_prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    latency = time.perf_counter() - start
    return output, latency

def call_ollama(model_name, prompt):
    import requests

    start = time.perf_counter()
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model_name,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.1},
        },
        timeout=120,
    )
    response.raise_for_status()
    latency = time.perf_counter() - start
    return response.json().get("response", ""), latency


def call_model(provider, model_name, prompt):
    if provider == "gemini":
        return call_gemini(model_name, prompt)
    if provider == "huggingface":
        return call_huggingface(model_name, prompt)
    if provider == "hf_local":
        return call_hf_local(model_name, prompt)
    if provider == "ollama":
        return call_ollama(model_name, prompt)
    raise ValueError(f"Unsupported provider: {provider}")

## 6. Run Experiments

Experiment size: 2 models × 3 prompt strategies × 30 posts = 180 model calls.

In [12]:
strategies = list(PROMPT_TEMPLATES.keys())
expected_calls = len(MODELS) * len(strategies) * len(test_set)
print("Expected calls:", expected_calls)

Expected calls: 180


In [13]:
def extract_json_list(text):
    if not isinstance(text, str):
        return []
    match = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not match:
        return []
    try:
        data = json.loads(match.group(0))
        return data if isinstance(data, list) else []
    except json.JSONDecodeError:
        return []


def build_gemini_batch_prompt(strategy, examples_df):
    posts_json = examples_df[["id", "post"]].to_dict(orient="records")
    strategy_instruction = {
        "zero_shot": "Classify each post directly using the labels.",
        "few_shot": "Use these examples as guidance: an excited Apple prize tweet is positive/Apple; an error-filled iPad app complaint is negative/iPad or iPhone App; an informational iPad design panel tweet is neutral/iPad.",
        "structured_json": "Be strict about valid JSON. Return no markdown and no explanation.",
    }[strategy]
    return (
        "You are classifying social media posts for business analysis.\n\n"
        f"Allowed sentiment labels: {', '.join(SENTIMENTS)}.\n"
        f"Allowed topic labels: {topic_list_text()}.\n\n"
        f"Strategy: {strategy_instruction}\n\n"
        "For each post, choose exactly one sentiment and one topic.\n"
        "Return ONLY a valid JSON array. Each item must have: id, sentiment, topic.\n\n"
        f"Posts:\n{json.dumps(posts_json, ensure_ascii=False)}"
    )


def call_gemini_batch(model_name, strategy, examples_df):
    prompt = build_gemini_batch_prompt(strategy, examples_df)
    raw_output, latency = call_gemini(model_name, prompt)
    parsed_list = extract_json_list(raw_output)
    by_id = {int(item.get("id")): item for item in parsed_list if isinstance(item, dict) and str(item.get("id", "")).isdigit()}
    results = {}
    for _, example in examples_df.iterrows():
        item = by_id.get(int(example["id"]), {})
        results[int(example["id"])] = {
            "raw_output": json.dumps(item, ensure_ascii=False) if item else raw_output,
            "pred_sentiment": normalize_label(item.get("sentiment", "")),
            "pred_topic": normalize_label(item.get("topic", "")),
            "latency_seconds": latency / max(len(examples_df), 1),
            "error": "" if item else "Could not parse this id from Gemini batch output",
        }
    return results


_HF_LOCAL_CLASSIFIERS = {}

def get_hf_local_classifier(model_name):
    from transformers import pipeline
    if model_name not in _HF_LOCAL_CLASSIFIERS:
        _HF_LOCAL_CLASSIFIERS[model_name] = pipeline("zero-shot-classification", model=model_name)
    return _HF_LOCAL_CLASSIFIERS[model_name]


def call_hf_local_batch(model_name, strategy, examples_df):
    classifier = get_hf_local_classifier(model_name)
    posts = examples_df["post"].tolist()
    hypothesis_template = {
        "zero_shot": "This post is about {}.",
        "few_shot": "Based on examples of social media business feedback, this post is about {}.",
        "structured_json": "The best structured classification label for this post is {}.",
    }[strategy]

    start_all = time.perf_counter()
    sentiment_results = classifier(
        posts,
        candidate_labels=SENTIMENTS,
        hypothesis_template=hypothesis_template,
        batch_size=8,
    )
    topic_results = classifier(
        posts,
        candidate_labels=TOPICS,
        hypothesis_template=hypothesis_template,
        batch_size=8,
    )
    total_latency = time.perf_counter() - start_all
    per_row_latency = total_latency / max(len(examples_df), 1)

    results = {}
    for idx, (_, example) in enumerate(examples_df.iterrows()):
        sent_result = sentiment_results[idx]
        topic_result = topic_results[idx]
        pred_sentiment = sent_result["labels"][0]
        pred_topic = topic_result["labels"][0]
        raw = json.dumps({
            "sentiment_labels": sent_result["labels"],
            "sentiment_scores": sent_result["scores"],
            "topic_labels": topic_result["labels"],
            "topic_scores": topic_result["scores"],
            "sentiment": pred_sentiment,
            "topic": pred_topic,
        }, ensure_ascii=False)
        results[int(example["id"])] = {
            "raw_output": raw,
            "pred_sentiment": pred_sentiment,
            "pred_topic": pred_topic,
            "latency_seconds": per_row_latency,
            "error": "",
        }
    return results

if RUN_API_CALLS:
    rows = []
    experiment_df = test_set.copy()
    existing_results = pd.read_csv(RESULTS_FILE) if RESULTS_FILE.exists() else pd.DataFrame()

    for model_info in MODELS:
        for strategy in strategies:
            if not existing_results.empty:
                cached = existing_results[
                    (existing_results["provider"] == model_info["provider"]) &
                    (existing_results["model"] == model_info["model"]) &
                    (existing_results["strategy"] == strategy)
                ].copy()
                cached_error = cached["error"].fillna("").astype(str).str.len().sum() if "error" in cached else 1
                if len(cached) == len(experiment_df) and cached_error == 0:
                    print(f"Using cached results for {model_info['model']} / {strategy}")
                    rows.extend(cached.to_dict(orient="records"))
                    continue

            print(f"Running {model_info['model']} / {strategy} on {len(experiment_df)} examples...")
            try:
                if model_info["provider"] == "gemini":
                    batch_results = call_gemini_batch(model_info["model"], strategy, experiment_df)
                elif model_info["provider"] == "hf_local":
                    batch_results = call_hf_local_batch(model_info["model"], strategy, experiment_df)
                else:
                    batch_results = {}
                    for _, example in experiment_df.iterrows():
                        prompt = build_prompt(strategy, example["post"])
                        raw_output, latency = call_model(model_info["provider"], model_info["model"], prompt)
                        parsed = extract_json(raw_output)
                        batch_results[int(example["id"])] = {
                            "raw_output": raw_output,
                            "pred_sentiment": normalize_label(parsed.get("sentiment", "")),
                            "pred_topic": normalize_label(parsed.get("topic", "")),
                            "latency_seconds": latency,
                            "error": "",
                        }
            except Exception as exc:
                batch_results = {
                    int(example["id"]): {
                        "raw_output": "",
                        "pred_sentiment": "",
                        "pred_topic": "",
                        "latency_seconds": None,
                        "error": str(exc),
                    }
                    for _, example in experiment_df.iterrows()
                }

            for _, example in experiment_df.iterrows():
                pred = batch_results[int(example["id"])]
                rows.append({
                    "id": example["id"],
                    "model": model_info["model"],
                    "provider": model_info["provider"],
                    "strategy": strategy,
                    "post": example["post"],
                    "true_sentiment": example["sentiment"],
                    "true_topic": example["topic"],
                    "raw_output": pred["raw_output"],
                    "pred_sentiment": pred["pred_sentiment"],
                    "pred_topic": pred["pred_topic"],
                    "latency_seconds": pred["latency_seconds"],
                    "error": pred["error"],
                })

    results = pd.DataFrame(rows)
    results.to_csv(RESULTS_FILE, index=False)
else:
    print("RUN_API_CALLS is False. Set it to True after adding your API key.")
    results = pd.DataFrame()

results.head()

Running gemini-flash-lite-latest / zero_shot on 30 examples...


/var/folders/dw/dk66f8js2bdgjnjf1p41w88w0000gn/T/ipykernel_81282/605522051.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Running gemini-flash-lite-latest / few_shot on 30 examples...


Running gemini-flash-lite-latest / structured_json on 30 examples...


Using cached results for typeform/distilbert-base-uncased-mnli / zero_shot
Using cached results for typeform/distilbert-base-uncased-mnli / few_shot
Using cached results for typeform/distilbert-base-uncased-mnli / structured_json


,id,model,provider,strategy,post,true_sentiment,true_topic,raw_output,pred_sentiment,pred_topic,latency_seconds,error
0,1,gemini-flash-lite-latest,gemini,zero_shot,This dude next to me is holding and using an i...,positive,iPad,"{""id"": 1, ""sentiment"": ""neutral"", ""topic"": ""iP...",neutral,iPad,0.062381,
1,2,gemini-flash-lite-latest,gemini,zero_shot,RT @mention I hope everyone has an awesome wee...,positive,Apple,"{""id"": 2, ""sentiment"": ""positive"", ""topic"": ""A...",positive,Apple,0.062381,
2,3,gemini-flash-lite-latest,gemini,zero_shot,RT @mention line moving fast! Rt @mention have...,positive,Apple,"{""id"": 3, ""sentiment"": ""positive"", ""topic"": ""A...",positive,Apple,0.062381,
3,4,gemini-flash-lite-latest,gemini,zero_shot,Last minute flight change to #sxsw means I can...,positive,iPad,"{""id"": 4, ""sentiment"": ""positive"", ""topic"": ""i...",positive,iPad,0.062381,
4,5,gemini-flash-lite-latest,gemini,zero_shot,YES RT @mention LoL U r gadgetzilla! Have fun!...,positive,iPad,"{""id"": 5, ""sentiment"": ""positive"", ""topic"": ""i...",positive,iPad,0.062381,


## 7. Evaluate Results

After running the model calls, this section calculates sentiment accuracy, topic accuracy, exact-match accuracy, format compliance, and average latency.

In [14]:
if RESULTS_FILE.exists() and results.empty:
    results = pd.read_csv(RESULTS_FILE)

if not results.empty:
    results["sentiment_correct"] = results["pred_sentiment"].str.lower() == results["true_sentiment"].str.lower()
    results["topic_correct"] = results["pred_topic"] == results["true_topic"]
    results["exact_match"] = results["sentiment_correct"] & results["topic_correct"]
    results["format_compliant"] = results["pred_sentiment"].isin(SENTIMENTS) & results["pred_topic"].isin(TOPICS)

    matrix = (
        results.groupby(["model", "strategy"])
        .agg(
            sentiment_accuracy=("sentiment_correct", "mean"),
            topic_accuracy=("topic_correct", "mean"),
            exact_match_accuracy=("exact_match", "mean"),
            format_compliance=("format_compliant", "mean"),
            avg_latency_seconds=("latency_seconds", "mean"),
            calls=("id", "count"),
        )
        .reset_index()
        .sort_values("exact_match_accuracy", ascending=False)
    )
    matrix.to_csv(MATRIX_FILE, index=False)
    display(matrix)
else:
    print("No model results yet. Run the experiment cell first.")

,model,strategy,sentiment_accuracy,topic_accuracy,exact_match_accuracy,format_compliance,avg_latency_seconds,calls
1,gemini-flash-lite-latest,structured_json,0.733333,0.900000,0.633333,1.0,0.069622,30
0,gemini-flash-lite-latest,few_shot,0.633333,0.900000,0.566667,1.0,0.085241,30
2,gemini-flash-lite-latest,zero_shot,0.666667,0.900000,0.566667,1.0,0.062381,30
5,typeform/distilbert-base-uncased-mnli,zero_shot,0.400000,0.433333,0.233333,1.0,0.110717,30
4,typeform/distilbert-base-uncased-mnli,structured_json,0.333333,0.400000,0.166667,1.0,0.068771,30
3,typeform/distilbert-base-uncased-mnli,few_shot,0.366667,0.300000,0.066667,1.0,0.075305,30


In [15]:
if not results.empty:
    failures = results[~results["exact_match"]].copy()
    display(failures[["id", "model", "strategy", "post", "true_sentiment", "pred_sentiment", "true_topic", "pred_topic", "raw_output"]].head(20))
else:
    print("No failures to analyze yet because no model results have been generated.")

,id,model,strategy,post,true_sentiment,pred_sentiment,true_topic,pred_topic,raw_output
0,1,gemini-flash-lite-latest,zero_shot,This dude next to me is holding and using an i...,positive,neutral,iPad,iPad,"{""id"": 1, ""sentiment"": ""neutral"", ""topic"": ""iP..."
5,6,gemini-flash-lite-latest,zero_shot,"bored at #SxSW, try this iphone app: {link}",positive,neutral,iPad or iPhone App,iPad or iPhone App,"{""id"": 6, ""sentiment"": ""neutral"", ""topic"": ""iP..."
7,8,gemini-flash-lite-latest,zero_shot,All this #sxsw gadget lust is rubbing off on m...,positive,neutral,iPad,iPad,"{""id"": 8, ""sentiment"": ""neutral"", ""topic"": ""iP..."
8,9,gemini-flash-lite-latest,zero_shot,"#GoogleDoodle #sxsw Google Doodles, simple on ...",positive,positive,Other Google product or service,Google,"{""id"": 9, ""sentiment"": ""positive"", ""topic"": ""G..."
9,10,gemini-flash-lite-latest,zero_shot,keep up with new on #Japan quake from iPhone a...,positive,neutral,iPhone,iPhone,"{""id"": 10, ""sentiment"": ""neutral"", ""topic"": ""i..."
13,14,gemini-flash-lite-latest,zero_shot,"Okay, fair enough lining up for new tech, but ...",negative,neutral,iPad,iPad,"{""id"": 14, ""sentiment"": ""neutral"", ""topic"": ""i..."
16,17,gemini-flash-lite-latest,zero_shot,RT @mention Novelty of iPad news apps fades fa...,negative,neutral,iPad or iPhone App,iPad or iPhone App,"{""id"": 17, ""sentiment"": ""neutral"", ""topic"": ""i..."
17,18,gemini-flash-lite-latest,zero_shot,Grrr..plancast stuff exported to Google calend...,negative,negative,Google,Other Google product or service,"{""id"": 18, ""sentiment"": ""negative"", ""topic"": ""..."
19,20,gemini-flash-lite-latest,zero_shot,Auntie's voxpop of popular #sxsw apps is worth...,negative,neutral,Android,Android,"{""id"": 20, ""sentiment"": ""neutral"", ""topic"": ""A..."
22,23,gemini-flash-lite-latest,zero_shot,Is music happening at #SXSW or is it just a la...,neutral,negative,Apple,Apple,"{""id"": 23, ""sentiment"": ""negative"", ""topic"": ""..."


## 8. Notes for Memo and Failure Analysis

Use the results matrix to identify the best model + prompt strategy. In `evaluation/failure_analysis.md`, group failures into patterns such as:

- Sentiment confusion on neutral posts
- Topic confusion between Apple/iPad/iPhone
- Invalid JSON or extra explanation
- Mixed-emotion posts where the model chose the wrong dominant sentiment

Your interpretation and failure analysis should be written in your own words.